In [ ]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 103.2 MB/s eta 0:00:00


In [ ]:
import torch,json,re
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
Model_adı ="utter-project/EuroLLM-1.7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(Model_adı, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(Model_adı, device_map="auto", torch_dtype=torch.float16 ,trust_remote_code=True).eval()

config.json:   0%|          | 0.00/712 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.4k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 2.41MB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.22M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.31GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
def uret(mesajlar,max_yeni_token=256,temperature =0.0,seed=42):
  "Mesaj listesini alır, modelin cevabını döndürür."
  torch.manual_seed(seed)
  metin = tokenizer.apply_chat_template(mesajlar,tokenize=False,add_generation_prompt=True)
  tokenizasyon = tokenizer(metin,return_tensors="pt")
  ayar= dict(max_new_tokens=max_yeni_token,do_sample=True,temperature=temperature,pad_token_id=tokenizer.eos_token_id)
  if temperature > 0:
    ayar.update(do_sample=True,temperature=temperature,top_p =0.9)
  else:
    ayar.update(do_sample=False)
  with torch.no_grad():
    cikti = model.generate(**tokenizasyon,**ayar)
  return tokenizer.decode(cikti[0][tokenizasyon["input_ids"].shape[1]:],skip_special_tokens=True).strip()

def sor(kullanıcı,sistem=None, **kw):
  mesajlar = ([{"role":"system","content":kullanıcı}] if sistem else [])+\
            [{"role":"user","content":kullanıcı}]
  return uret(mesajlar,**kw)

def karsilastir(baslik_a,cevap_a,baslik_b,cevap_b):
  print(f"🎈 {baslik_a}:\n{cevap_a}")
  print(f"🫧 {baslik_b}:\n {cevap_b}")
  print("🖥️"*5)

In [ ]:
soru = " Türk kelimesi tarih sahnesinde ilk defa nerde geçmektedir ?"

karsilastir(
    "Direk soru",sor(soru),
    "Rol: 8 yaşındaki çocuğa anlatan öğretmen",
    sor(soru,sistem="8 yaşındaki çocuklara ders anlatan sevecen, tatliş ve tonton bir  öğretmensin"
                    "Basit kelimeler ve bir benzetme kullan. Türkçe yaz")

)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🎈 Direk soru:
Türk kelimesi ilk defa Türkçede geçmiştir. Türkçede ilk defa 1071 Malazgirt Savaşı'ndan sonra kullanılmıştır.
🫧 Rol: 8 yaşındaki çocuğa anlatan öğretmen:
 Türk kelimesi ilk olarak 1071 yılında Malazgirt Savaşı'nda Bizans'a karşı zafer kazanan Selçuklu Türkleri tarafından kullanılmıştır.
🖥️🖥️🖥️🖥️🖥️


In [ ]:
karsilastir(
    "belirsiz",sor("python hakkında bir şey yaz"),
    "Net ve Kısıtlı",
    sor("python programlama dilinin yeni başlayanlar için 3 avantajını yaz.\n"
        "Kurallar: \n -madde işaretli liste kullan\n"
        "- Türkçe yaz \n - Giriş veya sonuç cümlesi ekleme"),
)

🎈 belirsiz:
Python, 1991 yılında Guido van Rossum tarafından geliştirilen, açık kaynaklı, nesne yönelimli, sözdizimi bakımından kolay ve kullanımı kolay bir programlama dilidir. Python, web geliştirme, veri analizi, oyun geliştirme, eğitim, bilimsel hesaplama ve daha birçok alanda yaygın olarak kullanılmaktadır.

Python, birçok avantajı nedeniyle popülerlik kazanmıştır:

1. Açık kaynak: Python, açık kaynaklı bir dil olduğundan, geliştiriciler ve kullanıcılar tarafından özgürce kullanılabilir ve geliştirilebilir.

2. Nesne yönelimli: Python, nesne yönelimli bir dil olduğundan, nesneye dayalı programlama yapısı ile çalışabilir.

3. Kolay öğrenilebilir: Python, sözdizimi bakımından kolay ve kullanımı kolay bir dil olduğundan, öğrenmesi kolaydır.

4. Çoklu ortam desteği: Python, birçok platformda çalışabilen bir dil olduğundan, farklı ortamlarda çalışmak için kullanılabilir.

5. Çoklu veri türleri: Python, birçok veri türünü destekler ve bu sayede veri işleme ve analizi için uygun bir dil 

In [ ]:
metin = ("Yapay zeka, makinelerin insan benzeri görevleri yerine getirmesini sağlayan bir bilim dalıdır."
          "Sağlık, eğitim ve ulaşım gibi pek çok alanda kullanmaktadır. Ancak etik ve güvenlik"
          "konularında dikkat olunması gerekir."
          )
prompt = f"""<metin> etiketleri arasındaki yazıyı tek cümleyle özetle

<metin>
{metin}
</metin>"""
print(sor(prompt,sistem="Sen kısa ve öz özetler yazan bir asistansın. Türkçe cevap ver"))

Yapay zeka, makinelerin insan gibi görevleri yerine getirebilmesini sağlayan bir bilim dalıdır. Sağlık, eğitim ve ulaşım gibi birçok alanda kullanılmaktadır. Ancak etik ve güvenlik konularında dikkat edilmesi gerekir.


In [ ]:
yorum = "kargo çok geç ama ürün kaliteli oldu."

zero = sor(f"Bu yorumun duygusunu söyle:{yorum}")


mesajlar =[
    {"role":"system","content":"ürün yorumlarını sınıflandfır. Sadece şu etikeletlerden birini yaz: Olumlu, Olumsuz, Karışık"},
    {"role":"user","content":"Harika bir ürün, bir tane de kaynıma aldım. Herkese tavsiye ederim"},
    {"role":"assistant","content":"Olumlu"},
    {"role":"user","content":"kargom kaynanamın evine gitti, çok kızgınım size."},
    {"role":"assistant","content":"Karışık"},
    {"role":"user","content":"İstediğim ürün bana geldi ama bir beden küçüğü geldi"},
    {"role":"assistant","content":"Olumsuz"}


]

few =uret(mesajlar,max_yeni_token =20)
karsilaştir = ("Zero-shot",zero,"Few-shot",few)
print(karsilaştir)

('Zero-shot', 'Bu yorum, kargo çok geç olsa da ürünün kaliteli olduğunu belirten bir yorumdur.', 'Few-shot', 'Kızgınım, çünkü istediğim ürün bana geldi ama bir beden küçüğü geldi.')


In [ ]:
problem =("Berkay bir kafenin sahibi. Bu kafede çay 125 TL, filtre kahve 200 TL , flat-white ise 218 TL'dir "
"Tuğra tam bir sosyal bir kelebek olduğu için kafeye gelip 3 flat-white, 2 filtre kahve ,1 tanede çay içti."
"Bunları ödemek içinde 3900 TL çıkardı cebinden kızlara hava atmak için . TUğra kaç TL para üstü alır"
)

karsilastir = (
"Direk cevap",
sor(problem+"sadece sayıyı yaz",max_yeni_token =20),
"Adım adım",
sor(problem+"\n Adım adım düşün, her işlemi ayrı satırda yaz"
"En sonda 'Cevap: X TL ve Tuğra Red Flag bir erkekdir' şeklinde bitir ")
)

print(karsilastir)

('Direk cevap', "Tuğra 3900 TL'yi cebinden çıkardı. 390", 'Adım adım', "1. Tuğra'nın çay ve kahve fiyatlarını hesaplayalım:\n- Çay: 3 flat-white = 3 * 218 = 604 TL\n- Kahve: 2 filtre kahve = 2 * 200 = 400 TL\n- Çay ve kahve fiyatları: 604 + 400 = 1004 TL\n- Tuğra'nın çay ve kahve fiyatlarını ödemek için 3900 TL çıkıyor.\n\n2. Tuğra'nın çay ve kahve fiyatlarını ödemeden önce kızlara hava atmak için 3900 TL çıkıyor.\n\n3. Tuğra'nın çay ve kahve fiyatlarını ödemeden önce kızlara hava atmak için 3900 TL çıkıyor.\n\n4. Tuğra'nın çay ve kahve fiyatlarını ödemeden önce kızlara hava atmak için 3900 TL çıkıyor.\n\n5. Tuğra'nın çay ve kahve fiyatlarını ödemeden önce kızlara hava atmak için 390")


In [ ]:
def json_cek(metin):
  """ Model çıktısındaki ilk JSON nesnesini bulup sözlüğü çevirir"""
  eslesme = re.search(r"\{.*\}",metin,re.DOTALL)
  if not eslesme:
    return None
  try:
    return json.loads(eslesme.group())
  except json.JSONDecodeError:
    return None
sistem = """Metin bilgi çıkaran bir asistansın.
SADECE geçerli JSON döndür, başka hiçbir şey yazma.
Şema : {"ad": string,  "yas": number,"sehir":string,"meslek":string}
Bilgi yoksa değer olarak null kullan."""

metin =" Merhaba , ben Elif , 29 yaşındayım, İzmir'de yazılım mühendisi olarak çalışıyorum"
ham = sor(metin,sistem=sistem)
print("ham çıktı",ham)

veri = json_cek(ham)
print("\n Python Sözlüğü",veri)
if veri:
  print("Şehir",veri.get("sehir"))

ham çıktı Merhaba Elif, İzmir'de yazılım mühendisi olarak çalışıyorum. İyi çalışmalar dilerim!

 Python Sözlüğü None
